# Bronze

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/durgashyam9849@gmail.com/consolidated_pipeline/setup/utilities

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source', 'products', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f's3://sportsbar-dem/{data_source}/*.csv'
print(base_path)

In [0]:
df = (
    spark.read.format('csv')
    .option('header', 'true')
    .option('inferSchema', 'true')
    .load(base_path)
    .withColumn('read_timestamp', F.current_timestamp())
    .select('*', '_metadata.file_name', '_metadata.file_size')
)

In [0]:
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
df.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

# Silver

In [0]:
df_bronze = spark.sql(f'select * from {catalog}.{bronze_schema}.{data_source}')
display(df_bronze.limit(10))

In [0]:
display(df_bronze.filter(F.col('product_name').contains('Protien')))

In [0]:
display(df_bronze.filter(F.col('product_id').contains('X')))

In [0]:
division_mapping = {
    "Energy Bars": "Nutrition Bars",
    "Protein Bars": "Nutrition Bars",
    "Granola & Cereals": "Breakfast Foods",
    "Recovery Dairy": "Dairy & Recovery",
    "Healthy Snacks": "Healthy Snacks",
    "Electrolyte Mix": "Hydration & Electrolytes"
}

In [0]:
def map_division(df, mapping):

    mapping_expr = F.create_map(
        *[F.lit(x) for pair in mapping.items() for x in pair]
    )

    return df.withColumn(
        "division",
        F.coalesce(
            mapping_expr[F.col("category")],
            F.lit("Other")
        )
    )

In [0]:
def transf_df(df):
    df_silver = df.withColumn("product_id_raw", F.col("product_id").cast("string"))
    df_silver = df_silver.withColumns({
        'product_name': F.regexp_replace(F.col('product_name'), '(?i)Protien', 'Protein'),
        'category': F.when(F.col("category").isNull(), None
                    ).otherwise(F.initcap(F.regexp_replace(F.col('category'), '(?i)protien', 'protein'))),
        'product_code': F.sha2(F.col('product_name').cast('string'), 256),
        "product_id": F.when(F.col("product_id_raw").rlike(r"^[0-9]+$"),
                    F.col("product_id_raw")).otherwise(F.lit('999999')),
        "product_id_status": F.when(F.col("product_id_raw").isNull(), "missing")
                    .when(F.col("product_id_raw").rlike(r"^[0-9]+$"),"valid").otherwise("invalid"),
        "variant": F.regexp_extract(F.col("product_name"), r"\((.*?)\)",1)
    })
    df_silver = df_silver.withColumnRenamed("product_name", "product")
    df_silver = map_division(df_silver, division_mapping)
    return df_silver


In [0]:
df_silver = transf_df(df_bronze)
display(df_silver.limit(6))

In [0]:
df_silver = df_silver.select('product_code', 'division', 'category', 'product', 'variant', 'product_id','read_timestamp', 'file_name', 'file_size', 'product_id_status','product_id_raw')

In [0]:
display(df_silver.limit(10))

In [0]:
df_silver.write\
.format("delta").option("delta.enableChangeDataFeed", "true").option("mergeSchema", "true").mode("overwrite")\
.saveAsTable(f"{catalog}.{silver_schema}.{data_source}")


# Gold

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")
df_gold = df_silver.select('product_code', 'product_id', 'division', 'category', 'product', 'variant')

In [0]:
display(df_gold.limit(10))

In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

# Merging child data with parent table

In [0]:
df_parent_products = DeltaTable.forName(spark, 'fmcg.gold.dim_products')
df_child_products = spark.sql(f"SELECT product_code, division, category, product, variant FROM fmcg.gold.sb_dim_products;")

In [0]:
display(df_child_products.show(5))

# Upsert Operation

In [0]:
df_parent_products.alias('target').merge(
    source = df_child_products.dropDuplicates(['product_code']).alias('source'),
    condition = "target.product_code = source.product_code"
).whenMatchedUpdate(
    set = {
        'division' : 'source.division',
        'category' : 'source.category',
        'product' : 'source.product',
        'variant' : 'source.variant'
    }
).whenNotMatchedInsert(
    values = {
        'product_code' : 'source.product_code',
        'division' : 'source.division',
        'category' : 'source.category',
        'product' : 'source.product',
        'variant' : 'source.variant'
    }
).execute()
